In [3]:
!pip install quandl
!pip install PyPortfolioOpt==1.5.5
!pip install arch
!pip install numpy

In [4]:
!pip install scikit-learn==1.2.2
import sklearn
sklearn.__version__

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 35.4 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.5.2
    Uninstalling scikit-learn-1.5.2:
      Successfully uninstalled scikit-learn-1.5.2


'1.2.2'

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy

In [6]:
!pip install yfinance==0.2.40
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

yfinance: pandas_datareader support is deprecated & semi-broken so will be removed in a future verison. Just use yfinance.


In [7]:
def modify_date(date):
    return date[:10]

In [8]:
def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

In [9]:
def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

In [10]:
def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

In [11]:
def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

In [12]:
def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

In [13]:
def output(weight):
    if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

    elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

    stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX[0])
    injected_result["oex"]=S2*OEX
    return injected_result

# Step 1 - Preselection

In [238]:
# Preselection parameters set up
method = "K_means_multiple"          # or avg or anyother method
stock_type = "S&P100"
end_swan = "Swan9"         #equals the last swan before the swan we are going to predict
start_swan="Swan5"
start = dt(2020,11,1)
end = dt(2022,1,3)
selected_stock_number = 19     #number of stock preselected
start_swan_date = dt(2022,1,3)
end_swan_date = dt(2023,1,1)
w1 = 0.17
w2 = 0.17
MA_days = 30
N = 7 # This parameter is set to help classification method like K-means

In [239]:
#read all stocks and their swan beta
if stock_type == "S&P100":
    portfolio_beta = pd.read_excel("All_SwanBeta100_edit.xlsx",index_col=0)
else:
    portfolio_beta = pd.read_excel("All_SwanBeta500.xlsx",index_col=0)

In [240]:
selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
portfolio_beta =portfolio_beta[ selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

In [241]:
portfolio_beta

,Swan-3,Swan-2,Swan-1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10a,Swan10b
Ticker,,,,,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.060138,1.278190
GM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,2.447747,1.934870
KMI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,-1.141071,-0.541752
GOOG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,0.875866,1.104899
GOOGL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,0.880849,1.117074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UNP,0.850695,2.127809,1.993802,1.088334,0.427848,0.418409,0.283910,1.275539,1.334344,1.354688,1.049957,1.188887,-0.200038,0.632258
USB,0.407208,0.375769,0.483245,0.176170,1.187362,1.307068,0.811824,1.672467,0.905422,0.995170,0.906766,1.495656,0.501194,0.825853
WFC,1.388900,1.261192,1.686449,0.558622,1.587780,1.233240,0.300985,1.698834,1.182443,0.995686,0.827634,1.470901,0.571633,1.078228


In [242]:
corr=portfolio_beta.corr()
selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
corr1={}
for i in range(selected_corr.shape[1]):
    for j in range(i+1,selected_corr.shape[0]):
        selected_corr.iloc[j,i]=np.nan

In [243]:
selected_corr

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9
Swan1,1.0,0.014,0.111657,0.050773,0.008306,0.104552,0.082139,0.084064,0.047612
Swan2,NaN,1.000,0.567583,0.343071,0.402635,0.397171,0.027890,0.296565,0.008002
Swan3,NaN,NaN,1.000000,0.151748,0.258994,0.435468,0.243654,0.224784,0.103439
Swan4,NaN,NaN,NaN,1.000000,-0.023466,0.144116,0.154501,0.221883,0.090316
Swan5,NaN,NaN,NaN,NaN,1.000000,0.444576,0.217781,0.207962,0.563052
Swan6,NaN,NaN,NaN,NaN,NaN,1.000000,0.301024,0.482688,0.403373
Swan7,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.187585,0.162732
Swan8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.231610
Swan9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000


In [244]:
avg=selected_corr.mean(axis=1)
weight_swan = avg/sum(avg)
selected_beata=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
#portfolio_beta["corr_beta"]=selected_beata@weight_swan
#portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

In [245]:
portfolio_beta

,Swan-3,Swan-2,Swan-1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10a,Swan10b
Ticker,,,,,,,,,,,,,,
CHTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.583170,0.649030,0.797374,0.939407,1.060138,1.278190
GM,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.026711,0.792127,0.190040,1.553517,2.447747,1.934870
KMI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.540529,1.230239,0.919927,1.592887,-1.141071,-0.541752
GOOG,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.916183,0.944312,0.976498,0.875866,1.104899
GOOGL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.803565,1.167641,0.870361,0.938163,0.979057,0.880849,1.117074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
UNP,0.850695,2.127809,1.993802,1.088334,0.427848,0.418409,0.283910,1.275539,1.334344,1.354688,1.049957,1.188887,-0.200038,0.632258
USB,0.407208,0.375769,0.483245,0.176170,1.187362,1.307068,0.811824,1.672467,0.905422,0.995170,0.906766,1.495656,0.501194,0.825853
WFC,1.388900,1.261192,1.686449,0.558622,1.587780,1.233240,0.300985,1.698834,1.182443,0.995686,0.827634,1.470901,0.571633,1.078228


In [246]:
if method == "gamma":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
    portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
elif method =="average":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #Select 18 stocks
    portfolio_beta=portfolio_beta.sort_values(by=["avg"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

elif method == "K_means":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    Kmean = KMeans(n_clusters=N,random_state=1)#set the random_state to keep modeling the same every run
    Kmean.fit(selected_beta)
    ct = Kmean.cluster_centers_
    labels = Kmean.labels_
    current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
    print(current_dbscore)
    selected_beta["Group"]=Kmean.labels_
    print(selected_beta['Group'])
    res = ct.mean(axis=1)
    res [res<0]=0

    min_group = np.where(res==min(res))
    for i in range(N):
        Y_predic[selected_beta["Group"] == i] = res [i]
    selected_beta ["avg"]=Y_predic.values
    selected_beta=selected_beta.sort_values(by=["avg"])
    first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
    if first_group.shape[0] <10:
        second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
        preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
    else:
        preselected_stock = first_group

# Use results from multiple clusters
elif method=="K_means_multiple":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    seed_dict={}

    # Run 1000 trials with davies bouldin index
    for random in range(1,1001):
      Kmean = KMeans(n_clusters=N,random_state=random,n_init=10)
      Kmean.fit(selected_beta)
      labels = Kmean.labels_
      # Evaluate the davies-bouldin score
      dbscore=metrics.davies_bouldin_score(selected_beta, labels)
      seed_dict[random]=dbscore
      sorted_dict=sorted(seed_dict.items(), key=lambda x:x[1])

    # Put all stocks into a stock list
    stock_list=[]
    max_length=0
    for index in range(0,5):
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Kmean = KMeans(n_clusters=N,random_state=sorted_dict[index][0],n_init=10)
      Kmean.fit(selected_beta)
      ct = Kmean.cluster_centers_
      Kmean.labels_
      selected_beta["Group"]=Kmean.labels_
      res = ct.mean(axis=1)
      res [res<0]=0
      min_group = np.where(res==min(res))
      for i in range(N):
          Y_predic[selected_beta["Group"] == i] = res [i]
      selected_beta ["avg"]=Y_predic.values
      selected_beta=selected_beta.sort_values(by=["avg"])
      first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
      if first_group.shape[0] <10:
          second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
          preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
      else:
          preselected_stock = first_group
      if preselected_stock.shape[0]>max_length:
        max_length=preselected_stock.shape[0]
      stock_list.extend(preselected_stock.index.values)
    print(stock_list)

    # Count the times each stock appear
    values,counts=np.unique(stock_list,return_counts=True)
    count_dict={}
    preselected_stock=[]
    for t in range(0,len(values)):
      count_dict[values[t]]=counts[t]
    sorted_count=sorted(count_dict.items(), key=lambda x:x[1],reverse=True)
    print(sorted_count)

    for p in range(0,min(22,len(sorted_count))):
      if sorted_count[p][1]>0:
        preselected_stock.append(sorted_count[p][0])

    preselected_stock = sorted(preselected_stock)
    selected_beta=selected_beta.loc[preselected_stock]
    selected_beta["avg"]=selected_beta.loc[:,start_swan:end_swan].mean(axis=1)
    # selected_beta['avg']=selected_beta['avg'].mean()
    preselected_stock = selected_beta
    print(preselected_stock)

<ipython-input-246-c9fbb0f953c0>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-246-c9fbb0f953c0>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
<ipython-input-246-c9fbb0f953c0>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selecte

['KO', 'VZ', 'SO', 'EXC', 'SBUX', 'LLY', 'MCD', 'AMT', 'MRK', 'DUK', 'NEE', 'MDLZ', 'PEP', 'PG', 'MRK', 'SBUX', 'LLY', 'MCD', 'AMT', 'EXC', 'KO', 'MDLZ', 'DUK', 'PEP', 'PG', 'SO', 'VZ', 'NEE', 'PG', 'AMT', 'MCD', 'KO', 'VZ', 'NEE', 'MDLZ', 'SO', 'LLY', 'MRK', 'SBUX', 'EXC', 'DUK', 'PEP', 'PG', 'AMT', 'MCD', 'KO', 'VZ', 'NEE', 'MDLZ', 'SO', 'LLY', 'MRK', 'SBUX', 'EXC', 'DUK', 'PEP', 'PG', 'AMT', 'MCD', 'KO', 'VZ', 'NEE', 'MDLZ', 'SO', 'LLY', 'MRK', 'SBUX', 'EXC', 'DUK', 'PEP']
[('AMT', 5), ('DUK', 5), ('EXC', 5), ('KO', 5), ('LLY', 5), ('MCD', 5), ('MDLZ', 5), ('MRK', 5), ('NEE', 5), ('PEP', 5), ('PG', 5), ('SBUX', 5), ('SO', 5), ('VZ', 5)]
           Swan5     Swan6     Swan7     Swan8     Swan9  Group       avg
Ticker                                                                   
AMT     0.583651  0.050751  0.583639 -0.336659  0.882155      6  0.352708
DUK     0.771714 -0.322793  0.233701 -0.356250  1.165454      6  0.298365
EXC     0.794675  0.263389  0.605246 -0.123438  1.254518

<ipython-input-246-c9fbb0f953c0>:71: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]


In [247]:
ct

array([[ 0.67354281,  0.45562002,  0.72514199,  0.90129198,  0.77363664],
       [ 1.60270556,  2.05398818,  1.14866016,  1.44286565,  1.61349505],
       [ 0.69249712,  0.84826667,  1.57641749,  1.30236048,  1.29660361],
       [ 1.20045682,  1.2496375 ,  0.85160097,  0.89761825,  1.26619451],
       [-0.51356172,  3.47395844,  0.61286879,  1.90242395,  0.21285984],
       [ 0.63571696,  1.25106235, -0.25340007,  2.78743002,  1.02674747],
       [ 0.69687375,  0.22200862,  0.55807843, -0.10467753,  0.93008431]])

In [248]:
res

array([0.70584669, 1.57234292, 1.14322907, 1.09310161, 1.13770986,
       1.08951135, 0.46047352])

In [249]:
preselected_stock.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14 entries, AMT to VZ
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Swan5   14 non-null     float64
 1   Swan6   14 non-null     float64
 2   Swan7   14 non-null     float64
 3   Swan8   14 non-null     float64
 4   Swan9   14 non-null     float64
 5   Group   14 non-null     int32  
 6   avg     14 non-null     float64
dtypes: float64(6), int32(1)
memory usage: 840.0+ bytes


In [250]:
#portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan]*weight

In [251]:
preselected_stock["PR"]=1-preselected_stock["avg"]
preselected_stock.to_csv('preselected_stocks.csv')
preselected_stock

,Swan5,Swan6,Swan7,Swan8,Swan9,Group,avg,PR
Ticker,,,,,,,,
AMT,0.583651,0.050751,0.583639,-0.336659,0.882155,6,0.352708,0.647292
DUK,0.771714,-0.322793,0.233701,-0.356250,1.165454,6,0.298365,0.701635
EXC,0.794675,0.263389,0.605246,-0.123438,1.254518,6,0.558878,0.441122
KO,0.536675,0.325968,0.614677,-0.049526,1.161009,6,0.517761,0.482239
LLY,0.920577,0.280671,0.780342,0.061466,0.490481,6,0.506707,0.493293
MCD,0.321829,0.130072,0.483323,-0.202955,1.142392,6,0.374932,0.625068
MDLZ,0.828315,0.344205,0.089729,0.386481,0.928976,6,0.515541,0.484459
MRK,0.756485,0.674926,0.995745,-0.019569,0.582136,6,0.597945,0.402055
NEE,0.503894,0.415367,0.192033,-0.026635,1.062981,6,0.429528,0.570472


# Step 2 - Download data and adjuste price based on Swan Beta

In [252]:
#next line is used to input preselected stock from excel, if you are going to preselect by your self, use this blocks and note next one.
#selected_stock_list =  pd.read_excel("19_stickers.xlsx",header = None).values[:,0]
#swan_beta=pd.read_excel("19_stickers.xlsx",header = None,index_col=0).iloc[:,0]

In [253]:
#Set the paramter to download tradeinfo from Yahoo Finance
selected_stock=preselected_stock
selected_stock_list=selected_stock.index.values
swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
print(selected_stock_list)

['AMT' 'DUK' 'EXC' 'KO' 'LLY' 'MCD' 'MDLZ' 'MRK' 'NEE' 'PEP' 'PG' 'SBUX'
 'SO' 'VZ']


In [254]:
#download data from yahoo finance
yahooDataFrame= pdr.get_data_yahoo(list(selected_stock_list), start=start, end=end)["Adj Close"]


[*********************100%%**********************]  14 of 14 completed


In [255]:
yahooDataFrame

Ticker,AMT,DUK,EXC,KO,LLY,MCD,MDLZ,MRK,NEE,PEP,PG,SBUX,SO,VZ
Date,,,,,,,,,,,,,,
2020-11-02,209.372543,79.874260,25.390036,43.053295,125.409851,194.061829,49.554577,64.807388,68.311371,120.631424,126.179977,78.969681,50.379230,45.706928
2020-11-03,210.036087,81.321587,26.326046,43.699715,124.790535,197.932846,49.627529,64.984802,69.120590,122.318420,128.658020,81.192642,51.104912,45.746544
2020-11-04,212.582794,79.644409,26.171074,43.540318,141.568405,196.170822,50.101738,68.110687,66.665649,123.050339,128.129593,82.478630,51.019531,45.326698
2020-11-05,215.138397,80.214821,26.697968,43.779408,138.643478,197.485519,50.740097,67.975517,67.965874,123.219917,129.714874,83.241066,51.369572,46.063393
2020-11-06,217.138092,79.550758,26.555399,43.788258,135.994843,197.713715,50.922485,67.891045,68.847832,123.514465,130.489227,83.268600,51.190289,46.364414
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,262.766388,92.338211,35.454453,53.992294,270.657471,251.992249,61.056458,70.810928,85.063202,158.643555,151.273712,107.156677,60.177555,43.621735
2021-12-28,265.276733,93.482491,35.971935,54.204018,268.732666,251.654037,61.513088,71.032867,85.584846,159.466980,152.086288,108.423187,60.869560,43.654858
2021-12-29,265.635254,93.652351,36.644672,54.268459,270.667236,252.227142,61.848568,71.162346,85.687294,160.031342,153.347137,109.183090,61.175121,43.356766


In [256]:
#set parameters to calculate predict market drop
market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
#swan_to_predict=100
total_day=yahooDataFrame.shape[0]
t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
#t2=total_day-t1
t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
t0=int(total_day-t1-t2)

In [257]:
#calculate predict market drop
md = [] #Make market drop data list named md


j = 0 #Start from 0
inc = dd/(t1-1) #Calculate unit increasing market drop
for i in range(t1): #Append market drop data for t1
    md.append(round(j,4))
    j = j + inc

base = (t2+1)**(1/(dd*100)) #Calculate log base
l1 = [] #Make decreasing market drop list
for i in range(t2):
    l1.append(math.log((i+2),base)/100)

for i in range(t2): #Append market drop data for t2
    md.append(round((dd-l1[i]),4))

for i in range(t0):
    md.append(0)

market_drop = pd.DataFrame(md) #Change list into dataframe
#market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
market_drop1 = pd.read_csv("trainingMarketdrop.csv")

In [258]:
market_drop.to_csv("market_drop.csv")

In [259]:
market_drop

,0
0,0.0000
1,0.0045
2,0.0089
3,0.0134
4,0.0179
...,...
289,0.0000
290,0.0000
291,0.0000
292,0.0000


In [260]:
swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

In [261]:
yahooDataFrame


Ticker,AMT,DUK,EXC,KO,LLY,MCD,MDLZ,MRK,NEE,PEP,PG,SBUX,SO,VZ
Date,,,,,,,,,,,,,,
2020-11-02,209.372543,79.874260,25.390036,43.053295,125.409851,194.061829,49.554577,64.807388,68.311371,120.631424,126.179977,78.969681,50.379230,45.706928
2020-11-03,209.702721,81.212401,26.259837,43.597898,124.505990,197.598895,49.512396,64.809944,68.986989,122.016250,128.360004,80.971853,51.034711,45.680463
2020-11-04,211.915476,79.432917,26.040899,43.339681,140.929975,195.516220,49.871855,67.748222,66.410800,122.449137,127.542603,82.035041,50.880922,45.197204
2020-11-05,214.121593,79.894114,26.498027,43.475666,137.702106,196.493334,50.389572,67.430866,67.574684,122.313490,128.820158,82.567017,51.159448,45.865255
2020-11-06,215.767198,79.125898,26.289741,43.382432,134.761361,196.386801,50.452563,67.164393,68.318492,122.300746,129.286911,82.367893,50.910580,46.098009
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-12-27,262.766388,92.338211,35.454453,53.992294,270.657471,251.992249,61.056458,70.810928,85.063202,158.643555,151.273712,107.156677,60.177555,43.621735
2021-12-28,265.276733,93.482491,35.971935,54.204018,268.732666,251.654037,61.513088,71.032867,85.584846,159.466980,152.086288,108.423187,60.869560,43.654858
2021-12-29,265.635254,93.652351,36.644672,54.268459,270.667236,252.227142,61.848568,71.162346,85.687294,160.031342,153.347137,109.183090,61.175121,43.356766


In [262]:
yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
selected_stock_list=yahooDataFrame.columns.values
selected_stock_number = yahooDataFrame.shape[1]
selected_stock_list

array(['AMT', 'DUK', 'EXC', 'KO', 'LLY', 'MCD', 'MDLZ', 'MRK', 'NEE',
       'PEP', 'PG', 'SBUX', 'SO', 'VZ'], dtype=object)

# Step3 - MVO

In [263]:
#Set the paramter for MVO process
Portfolio_Date=end

In [264]:
#data cleaning - filling missing data in yahooDataFrame
data_cleaning(yahooDataFrame)

In [265]:
#calculated rf
#bond_yield = quandl.get('FRED/DGS3MO',api_key='X4EQzCpMfeaTCTDthRH_')
#bond_yield = quandl.get('FRED/DGS3MO')
#bond_yield=np.log(bond_yield/100+1)
#date=list(map(str,bond_yield.index.values))
#date=list(map(modify_date,date))
#week=pd.to_datetime(date).strftime("%W").values
#week_1=np.roll(week,1)
#week_difference=np.where(week!=week_1)
#week_difference=np.delete(week_difference,0,None)
#rf_week=bond_yield.iloc[week_difference-1]

In [266]:
# Calculated R for 1 year and 0.5 year
today_price = yahooDataFrame.iloc[-MA_days:,:]
one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
one_year_ago_date = one_year_ago_data.index.values[-1]
half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
half_year_ago_date = half_year_ago_data.index.values[-1]
one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)



In [267]:
# Calculate MA
MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
MA_Data=MA_Data.loc[one_year_ago_date:,:]
MA_now = MA_Data.iloc[-1,:]
MA_one_year_ago = MA_Data.iloc[0,:]
MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

In [268]:
# Calculate daily return
daily_return = yahooDataFrame.pct_change().dropna()
log_daily_return = np.log(1+daily_return)

# Get week end stock price data
stock_date = yahooDataFrame.index.values
week=pd.to_datetime(stock_date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
# Calculate weekly return

weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
log_weekly_return = np.log(1+weekly_return)

# Get month end stock price data
stock_date = yahooDataFrame.index.values
month=pd.to_datetime(stock_date).strftime("%m").values
month_1=np.roll(month,1)
month_difference=np.where(month!=month_1)
month_difference=np.delete(month_difference,0,None)
# Calculate monthly return
monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
log_monthly_return = np.log(1+monthly_return)

# Get semi-month end stock price data
semi_month = []
for i in range(1,len(month_difference)):
    semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
    semi_month.append(month_difference[i]-1)
# Calculate semi-monthly return
semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
log_semi_monthly_return = np.log(1+semi_monthly_return)

In [269]:
# Calculated Max Dropdown for each stock
stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
mdd=[]
for column in stock_return_one_year.columns:
    mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
mar=R_mv_one_year/mdd.iloc[:,0]

In [270]:
#calculate weekly sd for each stock
one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
weekly_std= np.std(one_year_weekly_return)*np.sqrt(one_year_weekly_return.shape[0])

#Calculate garch_std sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
res_all = one_year_monthly_return - np.mean(one_year_monthly_return)
judge = one_year_monthly_return.shape[0]
for stock in one_year_monthly_return.columns:
    res=res_all[stock]
    h= het_arch(res)
    if judge < 100 or h ==0:
        garch_std = weekly_std

    else:#build Garch Model
        std = []
        if judge <=200:
            p = 0
            q = 6
        else:
            p = 1
            q = 1

        std.append(fit_garch(one_year_monthly_return[stock],p,q))
        garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

#Calculate monthly sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
monthly_std = np.std(one_year_monthly_return)*np.sqrt(12);

#Calculate semi_monthly sd for each stock
one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
semi_monthly_std = np.std(one_year_semi_monthly_return)*np.sqrt(24);


/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


In [271]:
# Calculated MaxDrop
DD_range = 90
total_range = 90
DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
max_DD=[-1]*selected_stock_number


if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
    DD_start = DD_start.index[-1]
    DD_end = DD_end.index[-1]
    while DD_start > DD_end :
        DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
        DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
        if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
            DD_start = DD_start.index[-1]
            DD_mid = DD_mid.index[-1]
            price = yahooDataFrame.loc[DD_start:DD_mid,:]
            for i in range(len(yahooDataFrame.columns)):
                max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
            total_range = total_range + 1
        else:
            break

In [272]:
# Read OPT-data
optdata = pd.read_excel("optdata_SP100_20220103_a.xlsx",index_col=0,sheet_name = 0)
P_current = MA_now
P_fwd = optdata.P_fwd.loc[P_current.index]
R_fwd=P_fwd/P_current-1
R_60 = P_current/MA_one_year_ago - 1
R_60_half = P_current/MA_half_year_ago - 1
R_MA_1ago = R_mv_one_year
R_MA_halfago = R_mv_half_year
Rfwd_adjusted = pd.Series(index=R_MA_1ago.index)
for i in R_fwd.index:
    if R_fwd[i] < 0.5*R_60[i]:
        Rfwd_adjusted[i] = R_60_half[i]
    elif R_fwd[i] > 1.3*R_60[i]:
        if 1.3*R_60[i] > 0.7*R_fwd[i]:
            Rfwd_adjusted[i] = 1.3*R_60[i]
        else:
            Rfwd_adjusted[i] = 0.7*R_fwd[i]
    else:
        Rfwd_adjusted[i] = R_fwd[i]
Rfwd_adjusted[Rfwd_adjusted<0]=0

In [273]:
price_now = today_price.iloc[-1,:]
optimizer = optdata.loc[P_current.index,:]
FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
preselected_stock = preselected_stock.loc[selected_stock_list,:]
preselected_stock["Rfwd_adjusted"]=Rfwd_adjusted.values
preselected_stock["std_f"]=weekly_std.values
preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
preselected_stock.to_csv("preselected_Stocks_info.csv")
p=FFC.MA_now
EPS_fwd=optimizer.EPS_fwd
EPS_ttm=optimizer.EPS_ttm
fall=FFC.Fall_MDD
FFC.to_csv("WEDataBank.csv")

PE=p/EPS_fwd
PEG= p/(EPS_fwd*(EPS_fwd/EPS_ttm-1))
PEG[PEG<0]=2000


In [274]:
preselected_stock

,Swan5,Swan6,Swan7,Swan8,Swan9,Group,avg,PR,Rfwd_adjusted,std_f,SR,PR_SR
Ticker,,,,,,,,,,,,
AMT,0.583651,0.050751,0.583639,-0.336659,0.882155,6,0.352708,0.647292,0.215776,0.215137,1.002970,0.648253
DUK,0.771714,-0.322793,0.233701,-0.356250,1.165454,6,0.298365,0.701635,0.178619,0.149473,1.194996,0.766998
EXC,0.794675,0.263389,0.605246,-0.123438,1.254518,6,0.558878,0.441122,0.566574,0.178939,3.166308,0.784937
KO,0.536675,0.325968,0.614677,-0.049526,1.161009,6,0.517761,0.482239,0.211183,0.178112,1.185680,0.525105
LLY,0.920577,0.280671,0.780342,0.061466,0.490481,6,0.506707,0.493293,0.226380,0.302514,0.748327,0.426728
MCD,0.321829,0.130072,0.483323,-0.202955,1.142392,6,0.374932,0.625068,0.137842,0.137027,1.005948,0.626924
MDLZ,0.828315,0.344205,0.089729,0.386481,0.928976,6,0.515541,0.484459,0.205559,0.143652,1.430956,0.579522
MRK,0.756485,0.674926,0.995745,-0.019569,0.582136,6,0.597945,0.402055,0.221287,0.237466,0.931869,0.388117
NEE,0.503894,0.415367,0.192033,-0.026635,1.062981,6,0.429528,0.570472,0.252439,0.204650,1.233517,0.633588


In [275]:
cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
E_2 = Rfwd_adjusted/fall
E_3 = Rfwd_adjusted/PEG
E_4 = Rfwd_adjusted
std =  np.std(np.exp(one_year_weekly_return)-1)*np.sqrt(52)


/usr/local/lib/python3.10/dist-packages/numpy/core/fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


# Optimization

In [276]:
w0 = 1/selected_stock_number*np.ones(selected_stock_number)
uplim_ini = np.zeros(selected_stock_number)
uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
for i in uplim3.index:
    if i in ["SPY","IWD"]:
        uplim3.loc[i,:]=0
low_lim =  np.zeros(selected_stock_number)
multi=lambda x: x.T @cov@x

In [277]:
ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
#ef.add_constraint(lambda w: w@E_2== 12)
raw_weights = ef.max_sharpe(0.01)
cleaned_weights = ef.clean_weights()
print(cleaned_weights)
ef.portfolio_performance(verbose=True)


OrderedDict([('AMT', 0.0), ('DUK', 0.04102), ('EXC', 0.17), ('KO', 0.0), ('LLY', 0.06485), ('MCD', 0.17), ('MDLZ', 0.10407), ('MRK', 0.01784), ('NEE', 0.01875), ('PEP', 0.0), ('PG', 0.07055), ('SBUX', 0.00636), ('SO', 0.16656), ('VZ', 0.17)])
Expected annual return: 2.8%
Annual volatility: 9.8%
Sharpe Ratio: 0.18


/usr/local/lib/python3.10/dist-packages/pypfopt/efficient_frontier/efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
  warnings.warn(


(0.027668779581408585, 0.09756251527268588, 0.18110213263797667)

In [278]:
weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

In [279]:
sd1 = np.sqrt(weight1@cov@weight1.T)
R1 = weight1.T@E_2/100
sharpe_ratio1 = (R1-0.01)/sd1
print("new_sharpe:",sharpe_ratio1)

new_sharpe: 0.18110213261821334


weight2 =   pd.read_excel("old_weight.xlsx",index_col=0)
weight2=np.array(weight2.loc[list(cleaned_weights),:])[:,0]

sd2 = np.sqrt(weight2@cov@weight2.T)
R2 = weight2.T@E_2/100
sharpe_ratio2 = (R2-0.01)/sd2
print("old_sharpe:",sharpe_ratio2)

In [280]:
weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
weight = weight[weight.weight >=0.030]
weight.weight = weight.weight/sum(weight.weight)
weight.to_csv("weights.csv")

See pyportfolioopt.readthedocs.io/en/latest/MeanVariance.html for more reference


In [281]:
if stock_type == "S&P100":
    OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

elif stock_type == "S&P500":
    OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed


In [282]:
stock_price.to_csv("tradeinfo.csv")
stock_price

Ticker,DUK,EXC,LLY,MCD,MDLZ,PG,SO,VZ
Date,,,,,,,,
2022-01-03,92.516998,36.916355,264.163483,252.311676,61.801723,152.142319,61.264996,43.423004
2022-01-04,92.445496,36.877548,259.380493,252.245941,62.251396,152.674698,61.264996,44.275898
2022-01-05,92.651100,36.851677,253.392044,251.005859,63.150730,153.365829,61.336899,44.731319
2022-01-06,92.499130,36.696423,252.108765,253.354431,63.122635,152.076981,61.264996,44.516029
2022-01-07,93.419922,36.638203,252.274048,250.883789,63.281898,151.992920,61.687393,45.454620
...,...,...,...,...,...,...,...,...
2022-12-23,95.677879,40.282478,362.543793,256.992493,63.995449,146.079849,67.021294,33.546482
2022-12-27,96.309593,40.385426,359.567749,256.291351,64.367386,147.352859,67.516434,34.280117
2022-12-28,95.594269,40.188881,359.902832,254.629700,63.737946,145.448135,66.937225,33.895832


In [283]:
start=stock_price.loc["2022-01-03"]
end = stock_price.loc["2022-10-12"]
stock_drop=abs(end/start-1)
market_drop = abs(OEX.loc["2022-10-12"]/OEX.loc["2022-01-03"]-1)

In [284]:
beta = stock_drop/market_drop
1-beta.T@weight

,0
weight,0.456133


In [285]:
stock_drop

,0
Ticker,
DUK,0.145598
EXC,0.096194
LLY,0.207415
MCD,0.102644
MDLZ,0.122992
PG,0.222711
SO,0.072015
VZ,0.282358


In [286]:
print(stock_price.head(1))
weight


Ticker            DUK        EXC         LLY         MCD       MDLZ  \
Date                                                                  
2022-01-03  92.516998  36.916355  264.163483  252.311676  61.801723   

Ticker              PG         SO         VZ  
Date                                          
2022-01-03  152.142319  61.264996  43.423004  


,weight
DUK,0.042861
EXC,0.177629
LLY,0.067760
MCD,0.177629
MDLZ,0.108740
PG,0.073716
SO,0.174035
VZ,0.177629


In [287]:
rest=1
extra=0
for i, row in weight.iterrows():
  if row['weight']>0.17:
    rest=rest-row['weight']
    extra=extra+(row['weight']-0.17)
    row['weight']=0.17

for i,row in weight.iterrows():
  if row['weight']<0.165:
    row['weight']=row['weight']+row['weight']/rest*extra
    a=1

weight.to_csv("weights.csv")
weight

,weight
DUK,0.046798
EXC,0.170000
LLY,0.073985
MCD,0.170000
MDLZ,0.118729
PG,0.080488
SO,0.170000
VZ,0.170000


In [288]:
a=stock_price@weight
b=1-a.loc["2022-10-12"]/a.loc["2022-01-03"]
1-b/market_drop

,0
weight,0.732369


In [289]:
injected_result = pd.DataFrame(index = stock_price.index)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T


In [290]:
portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
injected_result["portfolio_value"]=stock_price@S1.T
S2 = 10000/float(OEX[0])
injected_result["oex"]=S2*OEX
injected_result.to_csv("Injected_result.csv")


<ipython-input-290-fb66288e77f1>:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])


In [291]:
temp=injected_result.loc[["2022-01-03","2022-10-12"]]
Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
DD=abs(temp['oex'][1]/temp['oex'][0]-1)
PR=1-(Portfolio_DD/DD)
print(PR)
temp

0.5664969226740415


<ipython-input-291-18973d3e8135>:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
<ipython-input-291-18973d3e8135>:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  DD=abs(temp['oex'][1]/temp['oex'][0]-1)


,portfolio_value,oex
Date,,
2022-01-03,10000.000000,10000.000000
2022-10-12,8819.577997,7277.015862


In [292]:
allocation=pd.read_excel('allocation weight.xlsx',index_col='stock selected')
dfMVOstock=deepcopy(weight)
dfMVOstock['PR_SR']=''
dfMVOstock['SR']=''
for i in range(len(weight)):
    dfMVOstock.iloc[i,1]=preselected_stock.loc[weight.index[i],'PR_SR']
    dfMVOstock.iloc[i,2]=preselected_stock.loc[weight.index[i],'SR']
dfMVOstock_sort=dfMVOstock.sort_values(by='PR_SR',ascending=False)
postallocationweight=post_allocation(dfMVOstock_sort)
postallocationweight


<ipython-input-12-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<ipython-input-12-2b9c13413cb0>:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfMVOweight.iloc[i,5]=allocationweight[0]
<ipython-input-12-2b9c13413cb0>:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
<i

,weight,PR_SR,SR,equal,proportion,adjusted proportion,direct
VZ,0.170000,1.02349,2.272101,0.125,0.184592,0.17,0.17
EXC,0.170000,0.784937,3.166308,0.125,0.141568,0.143653,0.17
SO,0.170000,0.779614,1.259246,0.125,0.140608,0.142693,0.17
DUK,0.046798,0.766998,1.194996,0.125,0.138333,0.140417,0.17
MCD,0.170000,0.626924,1.005948,0.125,0.113069,0.115154,0.17
MDLZ,0.118729,0.579522,1.430956,0.125,0.10452,0.106605,0.05
PG,0.080488,0.556379,1.314615,0.125,0.100346,0.102431,0.05
LLY,0.073985,0.426728,0.748327,0.125,0.076963,0.079047,0.05


In [293]:
#draw type E graph for postallocation
MVOweight=postallocationweight['weight']
equalweight=postallocationweight['equal']
proportionweight=postallocationweight['adjusted proportion']
directweight=postallocationweight['direct']
output(MVOweight).to_csv('no_post_allocation_result.csv')
output(equalweight).to_csv('equal_weight_result.csv')
output(proportionweight).to_csv('proportional_weight_result.csv')
output(directweight).to_csv('direct_weight_result.csv')

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed
<ipython-input-13-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed
<ipython-input-13-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 comple

#